# LLM Evaluation 3 of 4: Localize Unsupported Claims

**Guiding question:** Which exact claim lacks support, and which evidence span should a
reviewer inspect?

This notebook owns claim splitting, supported/contradicted/unknown outcomes, entity and
relation gaps, evidence spans, and threshold selection. General judge bias belongs to Part 2;
confidence calibration and abstention policy belong to Part 4.


| Part | Complaint | Measured evidence |
| --- | --- | --- |
| 1 | Whole-answer similarity hides one bad clause | Atomic claim table |
| 2 | Shared entities do not verify the relation | Relation and entity gaps |
| 3 | A risk score without labels chooses nothing | Precision/recall threshold sweep |

Grounding asks whether supplied evidence supports a claim. It does not certify world truth.


In [ ]:
# Deterministic setup and the shared Riverside evidence fixture.
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from evaluation_shared import COURSE_CAVEAT, RIVERSIDE_CASES, normalized_terms, set_seed, tokenize

set_seed()
print(COURSE_CAVEAT)


## Part 1 - Split before scoring

Riverside's award answer contains a supported subject/year/prize and two unsupported
upgrades: `longlisted -> won` and an invented novel title. A whole-answer score can blur them.


In [ ]:
# Small claim splitter for the controlled fixture.
def split_claims(answer):
    pieces = re.split(r"[.;]|\band\b", answer)
    return [piece.strip() for piece in pieces if piece.strip()]

award = RIVERSIDE_CASES[1]
answer = award["fluent_wrong"]
claims = split_claims(answer)
display(pd.DataFrame({"claim": claims}))
print("Evidence:", award["context"])


In [ ]:
# Outcome rules expose support, contradiction, and missing evidence.
CONTRADICTIONS = {
    ("won", "longlisted"),
    ("automatic", "requires"),
    ("allows", "prohibited"),
}

def claim_outcome(claim, evidence):
    claim_terms, evidence_terms = set(normalized_terms(claim)), set(normalized_terms(evidence))
    for claim_term, evidence_term in CONTRADICTIONS:
        if claim_term in claim_terms and evidence_term in evidence_terms:
            return "contradicted"
    coverage = len(claim_terms & evidence_terms) / max(len(claim_terms), 1)
    return "supported" if coverage >= 0.6 else "unknown"

claim_rows = [
    {"claim": claim, "outcome": claim_outcome(claim, award["context"])}
    for claim in claims
]
claim_table = pd.DataFrame(claim_rows)
display(claim_table)


**What changed:** the answer is no longer one score. The relation upgrade becomes
`contradicted`; the novel title becomes `unknown`. Unknown means the evidence is silent, not
that the claim is globally false.


## Part 2 - Localize the reviewer target

Entity checks catch invented names, dates, quantities, and works. They still miss a wrong
verb connecting otherwise correct entities, so relation checks remain separate.


In [ ]:
# Extract capitalized spans and years as inspectable entity candidates.
def entities(text):
    names = re.findall(r"\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b", text)
    years = re.findall(r"\b(?:19|20)\d{2}\b", text)
    return set(names + years)

entity_gap = entities(answer) - entities(award["context"])
relation_pairs = [
    {"claim_relation": "won", "evidence_relation": "longlisted", "match": False},
]
print("entity_gap=", sorted(entity_gap))
display(pd.DataFrame(relation_pairs))


In [ ]:
# Retain the evidence span and signal provenance for every claim.
localized_rows = []
for case in RIVERSIDE_CASES:
    for answer_type in ["candidate", "fluent_wrong"]:
        for claim in split_claims(case[answer_type]):
            outcome = claim_outcome(claim, case["context"])
            localized_rows.append({
                "case": case["id"],
                "answer_type": answer_type,
                "claim": claim,
                "evidence_span": case["context"],
                "outcome": outcome,
                "entity_gap": sorted(entities(claim) - entities(case["context"])),
            })
localized = pd.DataFrame(localized_rows)
display(localized[["case", "answer_type", "claim", "outcome", "entity_gap"]])


## Part 3 - Choose a threshold from labeled costs

The risk score below is a transparent proxy: contradiction is strongest, unknown is weaker,
and unsupported entities add evidence. Its threshold must be selected on labels.


In [ ]:
# Convert observed claim outcomes into a review-risk score.
def risk_score(row):
    base = {"supported": 0.05, "unknown": 0.55, "contradicted": 0.90}[row["outcome"]]
    return min(1.0, base + 0.10 * bool(row["entity_gap"]))

localized["risk"] = localized.apply(risk_score, axis=1)
localized["label"] = localized["answer_type"].eq("fluent_wrong")

threshold_rows = []
for threshold in np.linspace(0.1, 0.9, 9):
    predicted = localized["risk"] >= threshold
    true_positive = int((predicted & localized["label"]).sum())
    false_positive = int((predicted & ~localized["label"]).sum())
    false_negative = int((~predicted & localized["label"]).sum())
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    threshold_rows.append({
        "threshold": threshold, "precision": precision, "recall": recall,
        "review_rate": predicted.mean(),
    })
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(3))


In [ ]:
# Static threshold evidence: recall and review load move together.
ax = threshold_table.plot(
    x="threshold", y=["precision", "recall", "review_rate"],
    marker="o", figsize=(9, 4)
)
ax.set_ylim(0, 1.05)
ax.set_title("Claim-risk threshold trade-off")
ax.set_ylabel("rate")
plt.tight_layout()
plt.show()

eligible = threshold_table.query("recall >= 0.80")
chosen = eligible.sort_values(["review_rate", "threshold"]).iloc[0]
print("Chosen teaching threshold:", chosen.to_dict())


## Scorecard

- Whole answers became atomic claims.
- Claims received `supported`, `contradicted`, or `unknown` outcomes.
- Entity gaps and relation mismatches pointed to different reviewer targets.
- Evidence spans and signal provenance survived in every record.
- A labeled sweep chose a threshold from recall and review load.

The local rules are proxies, not calibrated NLI. Production validation must use labeled,
in-domain claims and authoritative evidence.

**Next:** [Part 4](04-calibration-and-confidence.ipynb) asks whether a confidence score has an
honest frequency meaning and how it should drive abstention.
